In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
import pandas as pd


# ============================================================
# CONFIGURACIÓN GENERAL
# ============================================================

INVALID_IDS_TSV = "/content/drive/MyDrive/TFG_Optuna/Individual_Optuna/optuna_individual_invalid_ids.tsv"


# ============================================================
# FUNCIONES AUXILIARES
# ============================================================

def read_invalid_ids(invalid_ids_tsv_path=None):
    if invalid_ids_tsv_path is None:
        return set()

    if not os.path.exists(invalid_ids_tsv_path):
        print(f"No se encontró el archivo de ids inválidos: {invalid_ids_tsv_path}")
        print("No se excluirán ids inválidos.")
        return set()

    df_invalid = pd.read_csv(invalid_ids_tsv_path, sep="\t")

    if "id" not in df_invalid.columns:
        raise ValueError("El TSV de ids inválidos no tiene columna 'id'.")

    invalid_ids = set(df_invalid["id"].dropna().astype(str).tolist())

    print(f"Ids inválidos cargados: {len(invalid_ids)}")

    return invalid_ids


def get_first_existing_column(df, candidates, required=True, label=""):
    for col in candidates:
        if col in df.columns:
            return col

    if required:
        raise ValueError(
            f"No se encontró ninguna columna válida para {label}. "
            f"Candidatas: {candidates}. "
            f"Columnas disponibles: {list(df.columns)}"
        )

    return None


def deduplicate_by_id(df, source_name):
    duplicated_count = df["id"].duplicated().sum()

    if duplicated_count > 0:
        print(f"{source_name}: hay {duplicated_count} ids duplicados. Se conserva la última fila por id.")
        df = df.drop_duplicates(subset=["id"], keep="last").copy()

    return df


def standardize_results_tsv(
    tsv_path,
    strategy_suffix,
    model_type,
    invalid_ids=None
):
    """
    Lee un TSV de resultados y lo convierte a un formato común.

    model_type:
        "individual"
        "global"
    """

    if invalid_ids is None:
        invalid_ids = set()

    if not os.path.exists(tsv_path):
        raise FileNotFoundError(f"No existe el archivo: {tsv_path}")

    df = pd.read_csv(tsv_path, sep="\t")

    if "id" not in df.columns:
        raise ValueError(f"El archivo {tsv_path} no tiene columna 'id'.")

    df["id"] = df["id"].astype(str)

    initial_rows = len(df)
    df = df[~df["id"].isin(invalid_ids)].copy()
    excluded_rows = initial_rows - len(df)

    if excluded_rows > 0:
        print(f"{strategy_suffix}: se excluyeron {excluded_rows} filas por ids inválidos.")

    df = deduplicate_by_id(df, strategy_suffix)

    if model_type == "individual":
        n_train_col = get_first_existing_column(
            df,
            ["n_train", "n_train_individual", "n_train_id"],
            label=f"n_train en {strategy_suffix}"
        )

        n_val_col = get_first_existing_column(
            df,
            ["n_val", "n_val_individual", "n_val_id"],
            label=f"n_val en {strategy_suffix}"
        )

        n_test_col = get_first_existing_column(
            df,
            ["n_test", "n_test_individual", "n_test_id"],
            label=f"n_test en {strategy_suffix}"
        )

    elif model_type == "global":
        n_train_col = get_first_existing_column(
            df,
            ["n_train_global", "n_train_global_total", "n_train"],
            label=f"n_train_global en {strategy_suffix}"
        )

        n_val_col = get_first_existing_column(
            df,
            ["n_val_global", "n_val_global_total", "n_val"],
            label=f"n_val_global en {strategy_suffix}"
        )

        n_test_col = get_first_existing_column(
            df,
            ["n_test_id", "n_test_global", "n_test", "n_test_global_total"],
            label=f"n_test_id en {strategy_suffix}"
        )

    else:
        raise ValueError("model_type debe ser 'individual' o 'global'.")

    column_map = {
        "id": "id",

        n_train_col: f"n_train_{strategy_suffix}",
        n_val_col: f"n_val_{strategy_suffix}",
        n_test_col: f"n_test_{strategy_suffix}",

        get_first_existing_column(df, ["n_features"], label=f"n_features en {strategy_suffix}"): f"n_features_{strategy_suffix}",
        get_first_existing_column(df, ["feature_block"], label=f"feature_block en {strategy_suffix}"): f"feature_block_{strategy_suffix}",
        get_first_existing_column(df, ["learning_rate", "best_learning_rate"], label=f"learning_rate en {strategy_suffix}"): f"learning_rate_{strategy_suffix}",
        get_first_existing_column(df, ["epochs", "trained_epochs_final_model"], label=f"epochs en {strategy_suffix}"): f"epochs_{strategy_suffix}",

        get_first_existing_column(df, ["MAE"], label=f"MAE en {strategy_suffix}"): f"MAE_{strategy_suffix}",
        get_first_existing_column(df, ["RMSE"], label=f"RMSE en {strategy_suffix}"): f"RMSE_{strategy_suffix}",
        get_first_existing_column(df, ["R2"], label=f"R2 en {strategy_suffix}"): f"R2_{strategy_suffix}",
        get_first_existing_column(df, ["MRE"], label=f"MRE en {strategy_suffix}"): f"MRE_{strategy_suffix}",

        get_first_existing_column(df, ["best_val_loss", "best_val_mae", "best_value_val_MAE"], label=f"best_val_loss en {strategy_suffix}"): f"best_val_loss_{strategy_suffix}",
        get_first_existing_column(df, ["last_train_loss"], label=f"last_train_loss en {strategy_suffix}"): f"last_train_loss_{strategy_suffix}",
        get_first_existing_column(df, ["last_val_loss", "last_val_mae"], label=f"last_val_loss en {strategy_suffix}"): f"last_val_loss_{strategy_suffix}",

        get_first_existing_column(df, ["y_test_real_vector"], label=f"y_test_real_vector en {strategy_suffix}"): f"y_test_real_vector_{strategy_suffix}",
        get_first_existing_column(df, ["y_test_pred_vector"], label=f"y_test_pred_vector en {strategy_suffix}"): f"y_test_pred_vector_{strategy_suffix}",
    }

    keep_original_cols = list(column_map.keys())

    df_std = df[keep_original_cols].rename(columns=column_map)

    final_cols = [
        "id",
        f"n_train_{strategy_suffix}",
        f"n_val_{strategy_suffix}",
        f"n_test_{strategy_suffix}",
        f"n_features_{strategy_suffix}",
        f"feature_block_{strategy_suffix}",
        f"learning_rate_{strategy_suffix}",
        f"epochs_{strategy_suffix}",
        f"MAE_{strategy_suffix}",
        f"RMSE_{strategy_suffix}",
        f"R2_{strategy_suffix}",
        f"MRE_{strategy_suffix}",
        f"best_val_loss_{strategy_suffix}",
        f"last_train_loss_{strategy_suffix}",
        f"last_val_loss_{strategy_suffix}",
        f"y_test_real_vector_{strategy_suffix}",
        f"y_test_pred_vector_{strategy_suffix}",
    ]

    df_std = df_std[final_cols]

    return df_std


def merge_pairwise_results(
    left_tsv_path,
    right_tsv_path,
    left_suffix,
    right_suffix,
    left_model_type,
    right_model_type,
    output_tsv_path,
    invalid_ids_tsv_path=None
):
    """
    Crea una tabla comparativa por pares.
    Hace inner merge por id.
    Excluye ids inválidos de Optuna.
    """

    invalid_ids = read_invalid_ids(invalid_ids_tsv_path)

    df_left = standardize_results_tsv(
        tsv_path=left_tsv_path,
        strategy_suffix=left_suffix,
        model_type=left_model_type,
        invalid_ids=invalid_ids
    )

    df_right = standardize_results_tsv(
        tsv_path=right_tsv_path,
        strategy_suffix=right_suffix,
        model_type=right_model_type,
        invalid_ids=invalid_ids
    )

    merged_df = pd.merge(
        df_left,
        df_right,
        on="id",
        how="inner"
    )

    metric_blocks = [
        "n_train",
        "n_val",
        "n_test",
        "n_features",
        "feature_block",
        "learning_rate",
        "epochs",
        "MAE",
        "RMSE",
        "R2",
        "MRE",
        "best_val_loss",
        "last_train_loss",
        "last_val_loss",
        "y_test_real_vector",
        "y_test_pred_vector",
    ]

    final_order = ["id"]

    for block in metric_blocks:
        final_order.append(f"{block}_{left_suffix}")
        final_order.append(f"{block}_{right_suffix}")

    merged_df = merged_df[final_order]

    output_dir = os.path.dirname(output_tsv_path)

    if output_dir != "":
        os.makedirs(output_dir, exist_ok=True)

    merged_df.to_csv(output_tsv_path, sep="\t", index=False)

    print("Merge completado.")
    print(f"Estrategia izquierda: {left_suffix}")
    print(f"Estrategia derecha: {right_suffix}")
    print(f"Filas izquierda: {len(df_left)}")
    print(f"Filas derecha: {len(df_right)}")
    print(f"Filas merge final: {len(merged_df)}")
    print(f"Guardado en: {output_tsv_path}")

    return merged_df

In [ ]:
# ============================================================
# RUTAS DE ENTRADA
# ============================================================

INDIVIDUAL_FIXED_TSV = "/content/drive/MyDrive/TFG_Results/individual_results.tsv"
GLOBAL_FIXED_TSV = "/content/drive/MyDrive/TFG_Results/global_results_by_id.tsv"

INDIVIDUAL_OPTUNA_TSV = "/content/drive/MyDrive/TFG_Optuna/Individual_Optuna/optuna_individual_results.tsv"
GLOBAL_OPTUNA_TSV = "/content/drive/MyDrive/TFG_Optuna/Global_Optuna/optuna_global_results_by_id.tsv"

INVALID_IDS_TSV = "/content/drive/MyDrive/TFG_Optuna/Individual_Optuna/optuna_individual_invalid_ids.tsv"


# ============================================================
# RUTA DE SALIDA
# ============================================================

PAIRWISE_OUTPUT_DIR = "/content/drive/MyDrive/TFG_Optuna/Pairwise_Comparisons"

os.makedirs(PAIRWISE_OUTPUT_DIR, exist_ok=True)

In [ ]:
merged_ind_fixed_global_fixed = merge_pairwise_results(
    left_tsv_path=INDIVIDUAL_FIXED_TSV,
    right_tsv_path=GLOBAL_FIXED_TSV,
    left_suffix="individual_fixed",
    right_suffix="global_fixed",
    left_model_type="individual",
    right_model_type="global",
    output_tsv_path=os.path.join(
        PAIRWISE_OUTPUT_DIR,
        "pairwise_individual_fixed_vs_global_fixed.tsv"
    ),
    invalid_ids_tsv_path=INVALID_IDS_TSV
)

merged_ind_fixed_global_fixed.head()

Ids inválidos cargados: 244
individual_fixed: se excluyeron 234 filas por ids inválidos.
global_fixed: se excluyeron 234 filas por ids inválidos.
Merge completado.
Estrategia izquierda: individual_fixed
Estrategia derecha: global_fixed
Filas izquierda: 141
Filas derecha: 141
Filas merge final: 141
Guardado en: /content/drive/MyDrive/TFG_Optuna/Pairwise_Comparisons/pairwise_individual_fixed_vs_global_fixed.tsv


,id,n_train_individual_fixed,n_train_global_fixed,n_val_individual_fixed,n_val_global_fixed,n_test_individual_fixed,n_test_global_fixed,n_features_individual_fixed,n_features_global_fixed,feature_block_individual_fixed,...,best_val_loss_individual_fixed,best_val_loss_global_fixed,last_train_loss_individual_fixed,last_train_loss_global_fixed,last_val_loss_individual_fixed,last_val_loss_global_fixed,y_test_real_vector_individual_fixed,y_test_real_vector_global_fixed,y_test_pred_vector_individual_fixed,y_test_pred_vector_global_fixed
0,2,309,149043,66,31938,66,66,2048,2233,fingerprint_only,...,24.530.112.743.377.600,0.064246,0.170801,0.04543,2.564.368.486.404.410,0.064246,"28.30000114,30.27500153,1.50000000,12.19999981...","28.30000114,30.27500153,1.50000000,12.19999981...","28.89887619,28.86660767,8.31398582,12.72268867...","29.17040634,28.29000664,10.79966736,18.6729087..."
1,4,158,149043,34,31938,34,34,2048,2233,fingerprint_only,...,0.11366395652294159,0.064246,0.005848,0.04543,0.11530713737010956,0.064246,"4.19999981,1.90000010,6.50000000,4.40000010,5....","4.19999981,1.90000010,6.50000000,4.40000010,5....","2.41138983,1.97228479,6.26480770,4.39401913,6....","2.40662003,1.64516211,4.99883556,3.26555538,4...."
2,7,152,149043,33,31938,32,32,2048,2233,fingerprint_only,...,0.10003945976495743,0.064246,0.006514,0.04543,0.10680156946182251,0.064246,"4.70000029,7.20000029,15.50000000,4.19999981,2...","4.70000029,7.20000029,15.50000000,4.19999981,2...","8.84502697,10.37789249,11.32661152,4.03639841,...","4.69789410,5.29721546,13.65775681,7.71619177,1..."
3,9,401,149043,86,31938,86,86,2048,2233,fingerprint_only,...,0.032434865832328796,0.064246,0.008942,0.04543,0.03413492441177368,0.064246,"0.55000019,1.44000006,0.19000053,1.12000036,0....","0.55000019,1.44000006,0.19000053,1.12000036,0....","0.76963711,4.55827332,-0.50823402,1.08895540,0...","0.48365879,2.21589184,-0.85505772,2.35377932,0..."
4,19,294,149043,63,31938,63,63,2048,2233,fingerprint_only,...,0.19499565660953522,0.064246,0.006473,0.04543,0.1974235624074936,0.064246,"3.59999990,4.50000000,7.09999990,3.90000010,4....","3.59999990,4.50000000,7.09999990,3.90000010,4....","4.39849615,9.32486820,8.07400608,9.43267155,4....","3.58994579,5.56482553,7.45460749,4.63468504,4...."


In [ ]:
merged_ind_fixed_ind_optuna = merge_pairwise_results(
    left_tsv_path=INDIVIDUAL_FIXED_TSV,
    right_tsv_path=INDIVIDUAL_OPTUNA_TSV,
    left_suffix="individual_fixed",
    right_suffix="individual_optuna",
    left_model_type="individual",
    right_model_type="individual",
    output_tsv_path=os.path.join(
        PAIRWISE_OUTPUT_DIR,
        "pairwise_individual_fixed_vs_individual_optuna.tsv"
    ),
    invalid_ids_tsv_path=INVALID_IDS_TSV
)

merged_ind_fixed_ind_optuna.head()

Ids inválidos cargados: 244
individual_fixed: se excluyeron 234 filas por ids inválidos.
Merge completado.
Estrategia izquierda: individual_fixed
Estrategia derecha: individual_optuna
Filas izquierda: 141
Filas derecha: 141
Filas merge final: 141
Guardado en: /content/drive/MyDrive/TFG_Optuna/Pairwise_Comparisons/pairwise_individual_fixed_vs_individual_optuna.tsv


,id,n_train_individual_fixed,n_train_individual_optuna,n_val_individual_fixed,n_val_individual_optuna,n_test_individual_fixed,n_test_individual_optuna,n_features_individual_fixed,n_features_individual_optuna,feature_block_individual_fixed,...,best_val_loss_individual_fixed,best_val_loss_individual_optuna,last_train_loss_individual_fixed,last_train_loss_individual_optuna,last_val_loss_individual_fixed,last_val_loss_individual_optuna,y_test_real_vector_individual_fixed,y_test_real_vector_individual_optuna,y_test_pred_vector_individual_fixed,y_test_pred_vector_individual_optuna
0,2,309,309,66,66,66,66,2048,2048,fingerprint_only,...,24.530.112.743.377.600,4.753730,0.170801,0.121441,2.564.368.486.404.410,4.946555,"28.30000114,30.27500153,1.50000000,12.19999981...","28.30000114,30.27500153,1.50000000,12.19999981...","28.89887619,28.86660767,8.31398582,12.72268867...","29.64572716,32.68737030,5.17661524,13.85618401..."
1,4,158,158,34,34,34,34,2048,2048,fingerprint_only,...,0.11366395652294159,1.126244,0.005848,0.009778,0.11530713737010956,1.265188,"4.19999981,1.90000010,6.50000000,4.40000010,5....","4.19999981,1.90000010,6.50000000,4.40000010,5....","2.41138983,1.97228479,6.26480770,4.39401913,6....","2.65050364,1.70016861,6.03695107,4.04797745,5...."
2,7,152,152,33,33,32,32,2048,2048,fingerprint_only,...,0.10003945976495743,1.154123,0.006514,0.018037,0.10680156946182251,1.523880,"4.70000029,7.20000029,15.50000000,4.19999981,2...","4.70000029,7.20000029,15.50000000,4.19999981,2...","8.84502697,10.37789249,11.32661152,4.03639841,...","8.82721996,9.96708107,11.46120453,3.80209780,1..."
3,9,401,401,86,86,86,86,2048,2048,fingerprint_only,...,0.032434865832328796,0.111277,0.008942,0.000981,0.03413492441177368,0.129303,"0.55000019,1.44000006,0.19000053,1.12000036,0....","0.55000019,1.44000006,0.19000053,1.12000036,0....","0.76963711,4.55827332,-0.50823402,1.08895540,0...","0.58238173,1.48868895,0.22255993,1.04670143,0...."
4,19,294,294,63,63,63,63,2048,2048,fingerprint_only,...,0.19499565660953522,1.732921,0.006473,0.027767,0.1974235624074936,1.869297,"3.59999990,4.50000000,7.09999990,3.90000010,4....","3.59999990,4.50000000,7.09999990,3.90000010,4....","4.39849615,9.32486820,8.07400608,9.43267155,4....","4.41528177,8.75054455,7.89934015,9.05621815,4...."


In [ ]:
merged_global_fixed_global_optuna = merge_pairwise_results(
    left_tsv_path=GLOBAL_FIXED_TSV,
    right_tsv_path=GLOBAL_OPTUNA_TSV,
    left_suffix="global_fixed",
    right_suffix="global_optuna",
    left_model_type="global",
    right_model_type="global",
    output_tsv_path=os.path.join(
        PAIRWISE_OUTPUT_DIR,
        "pairwise_global_fixed_vs_global_optuna.tsv"
    ),
    invalid_ids_tsv_path=INVALID_IDS_TSV
)

merged_global_fixed_global_optuna.head()

Ids inválidos cargados: 244
global_fixed: se excluyeron 234 filas por ids inválidos.
Merge completado.
Estrategia izquierda: global_fixed
Estrategia derecha: global_optuna
Filas izquierda: 141
Filas derecha: 141
Filas merge final: 141
Guardado en: /content/drive/MyDrive/TFG_Optuna/Pairwise_Comparisons/pairwise_global_fixed_vs_global_optuna.tsv


,id,n_train_global_fixed,n_train_global_optuna,n_val_global_fixed,n_val_global_optuna,n_test_global_fixed,n_test_global_optuna,n_features_global_fixed,n_features_global_optuna,feature_block_global_fixed,...,best_val_loss_global_fixed,best_val_loss_global_optuna,last_train_loss_global_fixed,last_train_loss_global_optuna,last_val_loss_global_fixed,last_val_loss_global_optuna,y_test_real_vector_global_fixed,y_test_real_vector_global_optuna,y_test_pred_vector_global_fixed,y_test_pred_vector_global_optuna
0,2,149043,149043,31938,31938,66,66,2233,2233,all_features_global_train,...,0.064246,0.686649,0.04543,0.016496,0.064246,0.699917,"28.30000114,30.27500153,1.50000000,12.19999981...","28.30000114,30.27500153,1.50000000,12.19999981...","29.17040634,28.29000664,10.79966736,18.6729087...","29.00573158,30.80035210,4.18885040,19.37874031..."
1,4,149043,149043,31938,31938,34,34,2233,2233,all_features_global_train,...,0.064246,0.686649,0.04543,0.016496,0.064246,0.699917,"4.19999981,1.90000010,6.50000000,4.40000010,5....","4.19999981,1.90000010,6.50000000,4.40000010,5....","2.40662003,1.64516211,4.99883556,3.26555538,4....","3.42374420,1.98905754,6.80571508,4.05332041,6...."
2,7,149043,149043,31938,31938,32,32,2233,2233,all_features_global_train,...,0.064246,0.686649,0.04543,0.016496,0.064246,0.699917,"4.70000029,7.20000029,15.50000000,4.19999981,2...","4.70000029,7.20000029,15.50000000,4.19999981,2...","4.69789410,5.29721546,13.65775681,7.71619177,1...","4.55960083,6.46304989,13.85059547,3.97808409,1..."
3,9,149043,149043,31938,31938,86,86,2233,2233,all_features_global_train,...,0.064246,0.686649,0.04543,0.016496,0.064246,0.699917,"0.55000019,1.44000006,0.19000053,1.12000036,0....","0.55000019,1.44000006,0.19000053,1.12000036,0....","0.48365879,2.21589184,-0.85505772,2.35377932,0...","0.79728556,1.18139935,-0.03648758,1.22815418,0..."
4,19,149043,149043,31938,31938,63,63,2233,2233,all_features_global_train,...,0.064246,0.686649,0.04543,0.016496,0.064246,0.699917,"3.59999990,4.50000000,7.09999990,3.90000010,4....","3.59999990,4.50000000,7.09999990,3.90000010,4....","3.58994579,5.56482553,7.45460749,4.63468504,4....","3.28989983,4.60638428,6.30206823,3.76098537,4...."


In [ ]:
merged_ind_optuna_global_optuna = merge_pairwise_results(
    left_tsv_path=INDIVIDUAL_OPTUNA_TSV,
    right_tsv_path=GLOBAL_OPTUNA_TSV,
    left_suffix="individual_optuna",
    right_suffix="global_optuna",
    left_model_type="individual",
    right_model_type="global",
    output_tsv_path=os.path.join(
        PAIRWISE_OUTPUT_DIR,
        "pairwise_individual_optuna_vs_global_optuna.tsv"
    ),
    invalid_ids_tsv_path=INVALID_IDS_TSV
)

merged_ind_optuna_global_optuna.head()

Ids inválidos cargados: 244
Merge completado.
Estrategia izquierda: individual_optuna
Estrategia derecha: global_optuna
Filas izquierda: 141
Filas derecha: 141
Filas merge final: 141
Guardado en: /content/drive/MyDrive/TFG_Optuna/Pairwise_Comparisons/pairwise_individual_optuna_vs_global_optuna.tsv


,id,n_train_individual_optuna,n_train_global_optuna,n_val_individual_optuna,n_val_global_optuna,n_test_individual_optuna,n_test_global_optuna,n_features_individual_optuna,n_features_global_optuna,feature_block_individual_optuna,...,best_val_loss_individual_optuna,best_val_loss_global_optuna,last_train_loss_individual_optuna,last_train_loss_global_optuna,last_val_loss_individual_optuna,last_val_loss_global_optuna,y_test_real_vector_individual_optuna,y_test_real_vector_global_optuna,y_test_pred_vector_individual_optuna,y_test_pred_vector_global_optuna
0,2,309,149043,66,31938,66,66,2048,2233,fingerprint_only_optuna,...,4.753730,0.686649,0.121441,0.016496,4.946555,0.699917,"28.30000114,30.27500153,1.50000000,12.19999981...","28.30000114,30.27500153,1.50000000,12.19999981...","29.64572716,32.68737030,5.17661524,13.85618401...","29.00573158,30.80035210,4.18885040,19.37874031..."
1,4,158,149043,34,31938,34,34,2048,2233,fingerprint_only_optuna,...,1.126244,0.686649,0.009778,0.016496,1.265188,0.699917,"4.19999981,1.90000010,6.50000000,4.40000010,5....","4.19999981,1.90000010,6.50000000,4.40000010,5....","2.65050364,1.70016861,6.03695107,4.04797745,5....","3.42374420,1.98905754,6.80571508,4.05332041,6...."
2,7,152,149043,33,31938,32,32,2048,2233,fingerprint_only_optuna,...,1.154123,0.686649,0.018037,0.016496,1.523880,0.699917,"4.70000029,7.20000029,15.50000000,4.19999981,2...","4.70000029,7.20000029,15.50000000,4.19999981,2...","8.82721996,9.96708107,11.46120453,3.80209780,1...","4.55960083,6.46304989,13.85059547,3.97808409,1..."
3,9,401,149043,86,31938,86,86,2048,2233,fingerprint_only_optuna,...,0.111277,0.686649,0.000981,0.016496,0.129303,0.699917,"0.55000019,1.44000006,0.19000053,1.12000036,0....","0.55000019,1.44000006,0.19000053,1.12000036,0....","0.58238173,1.48868895,0.22255993,1.04670143,0....","0.79728556,1.18139935,-0.03648758,1.22815418,0..."
4,19,294,149043,63,31938,63,63,2048,2233,fingerprint_only_optuna,...,1.732921,0.686649,0.027767,0.016496,1.869297,0.699917,"3.59999990,4.50000000,7.09999990,3.90000010,4....","3.59999990,4.50000000,7.09999990,3.90000010,4....","4.41528177,8.75054455,7.89934015,9.05621815,4....","3.28989983,4.60638428,6.30206823,3.76098537,4...."


In [ ]:
!ls -lh "/content/drive/MyDrive/TFG_Optuna/Pairwise_Comparisons"

total 4.7M
-rw------- 1 root root 1.4M Jun 25 17:16 pairwise_global_fixed_vs_global_optuna.tsv
-rw------- 1 root root 1.1M Jun 25 17:16 pairwise_individual_fixed_vs_global_fixed.tsv
-rw------- 1 root root 1.1M Jun 25 17:16 pairwise_individual_fixed_vs_individual_optuna.tsv
-rw------- 1 root root 1.4M Jun 25 17:16 pairwise_individual_optuna_vs_global_optuna.tsv
